# ZHAW Commute Survey: analysis of the class data

The answers of the class are stored in the central Google Sheet and are read here directly from the survey link.

The analysis answers two kinds of questions:
- **Chi-squared test of independence**: are two *categorical* variables associated (living situation vs. means of transport, gender vs. means of transport)?
- **Spearman correlation**: are two *numerical or ordinal* variables monotonically related (distance, travel time, costs, satisfaction, stress)?

With a class-sized sample, checking the assumptions of the chi-squared test is a central part of the analysis.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2, chi2_contingency, fisher_exact, spearmanr, pearsonr

# Plot style settings
plt.style.use('dark_background')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Get current working directory
print(os.getcwd())

## Read the survey data from the central Google Sheet

In [ ]:
SURVEY_URL = 'https://script.google.com/macros/s/AKfycbwecUh787yY-DxLBazOx-J9LY69JvpRU51NWrJZifoGtaFgslBet5FWHfh1NXoiACrmtQ/exec'

df = pd.read_csv(SURVEY_URL + '?format=csv')
print('Respondents:', len(df))
df.head()

## Data overview

In [ ]:
for col in ['campus', 'transport', 'travel_pass', 'gender', 'living']:
    print(df[col].value_counts(), '\n')

In [ ]:
df[['door_to_door_min', 'distance_km', 'days_per_week', 'monthly_cost_chf', 'age']].describe().round(1)

In [ ]:
# Median travel time, distance and costs per means of transport
df.groupby('transport')[['door_to_door_min', 'distance_km', 'monthly_cost_chf', 'satisfied']].median()

## Chi-squared test 1: living situation vs. means of transport

- H0: there is no association between the living situation and the means of transport
- H1: there is an association between the living situation and the means of transport
- Significance level alpha = 0.05

In [ ]:
alpha = 0.05

tab = pd.crosstab(df['living'], df['transport'])
stat, p, dof, expected = chi2_contingency(tab)

print(tab, '\n')
print('Expected frequencies:\n', pd.DataFrame(expected, index=tab.index, columns=tab.columns).round(2), '\n')
print(f'Test statistic: {stat:.4f}, dof: {dof}, critical value: {chi2.ppf(1 - alpha, dof):.4f}, p-value: {p:.4f}')
print('Smallest expected frequency:', expected.min().round(2))
print(f'Cells with expected frequency < 5: {(expected < 5).sum()} of {expected.size}')

### Check the assumption: expected frequencies
The chi-squared test is only reliable if (rule of thumb) all expected frequencies are at least 5, or at least 80 % of them with none below 1. If the table above does not meet this rule, its p-value must not be interpreted.

**Remedy: merge categories into a 2 × 2 table** that matches the research question. Hypothesis: students in the category *Other* (e.g. living with their parents) live further away and therefore take the train more often.

- living: *Other* vs. *Own flat / Shared apartment*
- transport: *Train* vs. *other means of transport*

For a 2 × 2 table, `chi2_contingency` applies Yates' continuity correction by default. **Fisher's exact test** does not rely on large expected frequencies at all and is the standard choice for small 2 × 2 tables.

In [ ]:
df['living_2'] = np.where(df['living'] == 'Other', 'Other', 'Own flat / Shared apartment')
df['train'] = np.where(df['transport'] == 'Train', 'Train', 'Other transport')

tab_2x2 = pd.crosstab(df['living_2'], df['train'])
stat, p, dof, expected = chi2_contingency(tab_2x2)  # with Yates' correction
odds_ratio, p_fisher = fisher_exact(tab_2x2)

print(tab_2x2, '\n')
print('Row percentages:\n', (pd.crosstab(df['living_2'], df['train'], normalize='index') * 100).round(1), '\n')
print('Expected frequencies:\n', pd.DataFrame(expected, index=tab_2x2.index, columns=tab_2x2.columns).round(2), '\n')
print(f'Chi-squared (Yates): {stat:.4f}, dof: {dof}, p-value: {p:.4f}')
print(f"Fisher's exact test: p-value: {p_fisher:.4f}, odds ratio: {odds_ratio:.2f}")

n = tab_2x2.to_numpy().sum()
print(f"Cramér's V (phi): {np.sqrt(chi2_contingency(tab_2x2, correction=False)[0] / n):.3f}")
print('Reject H0' if p_fisher < alpha else 'Do not reject H0')

## Chi-squared test 2: gender vs. means of transport

- H0: there is no association between gender and using public transport
- H1: there is an association between gender and using public transport

The means of transport are merged into *public transport* (train, tram, bus) vs. *car / bike / foot*.

In [ ]:
df['public'] = np.where(df['transport'].isin(['Train', 'Tram or bus']), 'Public transport', 'Car / bike / foot')

tab_gender = pd.crosstab(df['gender'], df['public'])
stat, p, dof, expected = chi2_contingency(tab_gender)
odds_ratio, p_fisher = fisher_exact(tab_gender)

print(tab_gender, '\n')
print('Smallest expected frequency:', expected.min().round(2))
print(f'Chi-squared (Yates): {stat:.4f}, dof: {dof}, p-value: {p:.4f}')
print(f"Fisher's exact test: p-value: {p_fisher:.4f}")
print('Reject H0' if p_fisher < alpha else 'Do not reject H0')

## Spearman correlation

Spearman's rank correlation is used because
- the Likert items (`satisfied`, `stressful`, `productive`) are ordinal,
- travel time, distance and costs are right-skewed with a few large values,
- it measures *monotonic* relationships, which need not be linear.

For each pair: H0: rho = 0 (no monotonic relationship), H1: rho ≠ 0.

In [ ]:
cols = ['door_to_door_min', 'distance_km', 'monthly_cost_chf', 'days_per_week', 'age', 'satisfied', 'stressful', 'productive']
rho_matrix = df[cols].corr(method='spearman')

plt.figure(figsize=(9, 7))
sns.heatmap(rho_matrix, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, cbar_kws={'label': "Spearman's rho"})
plt.title(f'Spearman correlations (n = {len(df)})')
plt.show()

In [ ]:
# Spearman's rho with p-value for the pairs of interest
pairs = [
    ('distance_km', 'door_to_door_min'),
    ('distance_km', 'monthly_cost_chf'),
    ('door_to_door_min', 'satisfied'),
    ('door_to_door_min', 'stressful'),
    ('door_to_door_min', 'productive'),
    ('satisfied', 'stressful'),
    ('age', 'door_to_door_min'),
]
results = []
for x, y in pairs:
    rho, p = spearmanr(df[x], df[y])
    results.append({'variable 1': x, 'variable 2': y, 'rho': round(rho, 3), 'p-value': round(p, 4),
                    'significant (alpha = 0.05)': p < alpha})
pd.DataFrame(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(df['distance_km'], df['door_to_door_min'], c='#5cc2cc', s=40)
axes[0].set_xlabel('Distance one way (km)')
axes[0].set_ylabel('Door to door one way (minutes)')
axes[0].set_title('Distance vs. travel time')

# Small random jitter so that equal Likert answers do not hide each other
jitter = np.random.default_rng(1).uniform(-0.15, 0.15, len(df))
axes[1].scatter(df['door_to_door_min'], df['satisfied'] + jitter, c='#e5a58f', s=40)
axes[1].set_xlabel('Door to door one way (minutes)')
axes[1].set_ylabel('I am satisfied with my commute (1-5)')
axes[1].set_yticks(range(1, 6))
axes[1].set_title('Travel time vs. satisfaction')

plt.tight_layout()
plt.show()

### Spearman vs. Pearson

In [ ]:
for x, y in [('distance_km', 'monthly_cost_chf'), ('door_to_door_min', 'satisfied')]:
    rho, p_s = spearmanr(df[x], df[y])
    r, p_p = pearsonr(df[x], df[y])
    print(f"{x} vs. {y}: Spearman's rho = {rho:.3f} (p = {p_s:.4f}), Pearson's r = {r:.3f} (p = {p_p:.4f})")

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')